In [15]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [16]:
df=pd.read_csv("clean_song.csv")

In [17]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2899 entries, 0 to 2898
Data columns (total 4 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   track_id     2899 non-null   int64
 1   track_name   2898 non-null   str  
 2   tags         2899 non-null   str  
 3   artwork_url  2899 non-null   str  
dtypes: int64(1), str(3)
memory usage: 624.6 KB


In [18]:
tfidf=TfidfVectorizer(max_features=1500,stop_words='english')
vector=tfidf.fit_transform(df['tags']).toarray()

In [19]:
similarity=cosine_similarity(vector)
similarity

array([[1.        , 0.        , 0.02593001, ..., 0.        , 0.        ,
        0.        ],
       [0.        , 1.        , 0.        , ..., 0.        , 0.        ,
        0.        ],
       [0.02593001, 0.        , 1.        , ..., 0.        , 0.        ,
        0.        ],
       ...,
       [0.        , 0.        , 0.        , ..., 1.        , 0.86957275,
        0.27419788],
       [0.        , 0.        , 0.        , ..., 0.86957275, 1.        ,
        0.238435  ],
       [0.        , 0.        , 0.        , ..., 0.27419788, 0.238435  ,
        1.        ]], shape=(2899, 2899))

In [20]:
df.columns

Index(['track_id', 'track_name', 'tags', 'artwork_url'], dtype='str')

In [21]:
def get_name_from_index(i):
    if i<len(df) and i>0:
        return df.loc[i,'track_name']
    else:
        return''

In [22]:
get_name_from_index(5)

'meant to be'

In [23]:
def get_index_from_name(name):
    clean_song=name.lower().strip().replace(" ","").replace("-","")
    match=df[df['track_name'].str.strip().str.replace(" ",'').str.replace("-","")==clean_song]
    if not match.empty:
        return match.index[0]
    return -1

In [24]:
get_index_from_name('meant to be')

5

In [25]:
name=input("enter song name:")
index=get_index_from_name(name)
if index!=-1:
    similarity_index=list(enumerate(similarity[index]))
    similarity_index=sorted(similarity_index,key=lambda x:x[1],reverse=True)
    print("You are listened:",df.loc[index,'track_name'])
    print("You must listen following song")

    for i in range(1,6):
        song_idn=similarity_index[i][0]
        print(i,':',get_name_from_index(song_idn))
else:
    print("song not found!")

enter song name: meant to be


You are listened: meant to be
You must listen following song
1 : dirt
2 : cruise
3 : this is how we roll feat luke bryan
4 : queen
5 : line of sight feat wynne  mansionair


In [26]:
similarity = similarity.astype("float32")

In [27]:
df.to_csv("song_data.csv",index=False)

In [28]:
import pickle
pickle.dump(similarity,(open("similarity.pkl","wb")))